In [17]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier   # Asegurarse de tener xgboost instalado: pip install xgboost
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split



In [24]:
# Cargamos datos
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

X = train.drop(columns=["Target_Risco"])
y = train["Target_Risco"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
X_test = test.drop(columns=["ID_Cliente"])

num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

train.head()



,ID_Cliente,Data_Solicitude,Idade,Lonxitude_Nome,Num_Fillos,Profesion,Anos_Emprego,Ingresos_Anuais,Tipo_Dispositivo,Tempo_Web_Minutos,...,Limite_Credito_Total,Cota_Mensual_Prestamos,Ratio_Cota_Ingresos,Prestamos_Activos,Antiguedade_Cliente_Anos,Saldo_Medio_3M,Variacion_Saldo_6M,Fondo_Emerxencia_Meses,Indice_Estres_Financeiro,Target_Risco
0,122279,2021-01-01,27,17,2,Autónomo,2.1,7660.67,Android,45.0,...,5660.66,60.30,0.094444,1,2.3,489.85,-0.091,8.11,0.234,0
1,121221,2021-01-01,44,19,1,Funcionario,17.4,126784.59,Windows,37.5,...,54104.76,754.26,0.071389,1,17.2,2820.78,-0.131,3.74,0.256,1
2,111934,2021-01-01,30,12,1,Asalariado,2.2,9629.39,Windows,30.5,...,4330.94,3.09,0.003850,0,0.0,231.86,-0.183,24.00,0.128,0
3,101906,2021-01-01,42,29,1,Desempregado,0.0,3332.75,Windows,NaN,...,5664.84,1.86,0.006695,0,7.1,221.12,-0.047,24.00,0.163,1
4,117317,2021-01-01,33,16,0,Desempregado,0.0,6993.01,Android,21.5,...,6614.56,6.69,0.011478,0,0.0,392.57,0.192,24.00,0.000,0


In [ ]:
# Preprocesado

In [19]:
# Modelo 1: Bootstraping - Bagging

def m1_bagging_RF(X_train, y_train, X_val, y_val):
    modelo1 = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42
    )

    modelo1.fit(X_train, y_train)

    preds1 = modelo1.predict(X_val)

    print("Modelo 1: Random Forest")
    f1_score = f1_score(y_val, preds1, average="weighted")
    accuracy_score = accuracy_score(y_val, preds1)
    print("Accuracy:", accuracy_score)
    print("F1:", f1_score)
    
    return modelo1, preds1, accuracy_score,f1_score


In [20]:
# Modelo 2: Boosting - XGBoost

def m2_boosting_XGB(X_train, y_train, X_val, y_val):
    modelo2 = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softmax",
        num_class=4,
        random_state=42,
        eval_metric="mlogloss"
    )

    modelo2.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=20, verbose=False)

    preds2 = modelo2.predict(X_val)

    print("Modelo 2: XGBoost")
    f1_score = f1_score(y_val, preds2, average="weighted")
    accuracy_score = accuracy_score(y_val, preds2)
    print("Accuracy:", accuracy_score)
    print("F1:", f1_score)
    
    return modelo2, preds2, accuracy_score,f1_score

In [22]:
# Compararar modelos

m1_bagging_RF, RF_preds, RF_accuracy, RF_f1 = m1_bagging_RF(X_train, y_train, X_val, y_val)
m2_boosting_XGB, XGB_preds, XGB_accuracy, XGB_f1 = m2_boosting_XGB(X_train, y_train, X_val, y_val)

resultados = pd.DataFrame({
    "Modelo": ["RandomForest", "XGBoost"],
    "Accuracy": [RF_accuracy, XGB_accuracy],
    "F1_weighted": [RF_f1, XGB_f1]
})

print(resultados.sort_values(by="F1_weighted", ascending=False))


ValueError: could not convert string to float: '2021-09-25'

In [25]:
# Prediccion sobre test
RF_test_preds = m1_bagging_RF.predict(X_test)
XGB_test_preds = m2_boosting_XGB.predict(X_test)

submission_RF = pd.DataFrame({
    "ID_Cliente": test["ID_Cliente"],
    "Target_Risco": RF_test_preds
})
submission_XGB = pd.DataFrame({
    "ID_Cliente": test["ID_Cliente"],
    "Target_Risco": XGB_test_preds
})

submission_RF.to_csv("submission_RF.csv", index=False)
submission_XGB.to_csv("submission_XGB.csv", index=False)

print("Predicciones guardadas en submission_RF.csv y submission_XGB.csv")

AttributeError: 'function' object has no attribute 'predict'